In [ ]:
import sys
import os

# localizar caminho automaticamente
import pandas as pd

from rdkit import RDConfig
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))

import sascorer
from rdkit import Chem

In [ ]:
# ===== CONFIGURAÇÕES =====
# Defina aqui os caminhos e parâmetros

# Diretório base
base_dir = '/Users/francisco/Documents/Scripts/DeNovo_HsDHODH/LigBuilder_Results/Build_HsDHODH_HIGHDIV_MAY/result/process_HsDHODH/'

# Arquivo de entrada
input_file = base_dir + 'dados_sem_duplicatas.csv'

# Arquivos de saída
output_file_completo = base_dir + 'dados_com_sa_score.csv'
output_file_filtrado = base_dir + 'dados_sa_score_filtrado.csv'

# Nome da coluna de SMILES
coluna_smiles = 'SMILES'

# Threshold do SA Score (moléculas com SA Score < threshold receberão valor 1)
threshold = 4.0
# =========================

In [ ]:
# Carregar o dataset
df = pd.read_csv(input_file)

print(f"Dataset carregado: {len(df)} linhas")
df.head()

In [ ]:
# Função para calcular o SA Score
def calculate_sa_score(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return sascorer.calculateScore(mol)
        else:
            return None
    except:
        return None

# Aplicar a função na coluna de SMILES
df['SA_Score'] = df[coluna_smiles].apply(calculate_sa_score)

# Criar coluna binária baseada no threshold
df['SA_Score_Binary'] = (df['SA_Score'] < threshold).astype(int)

# Visualizar resultado
df.head()

In [ ]:
# Contar quantos 1s e 0s na coluna binária
contagem = df['SA_Score_Binary'].value_counts().sort_index()
print(contagem)

In [ ]:
# Salvar dataset completo com SA Score
df.to_csv(output_file_completo, index=False)
print(f"Dataset completo salvo em: {output_file_completo}")

In [ ]:
# Filtrar apenas moléculas com SA_Score_Binary = 1 (SA Score < threshold)
df_filtered = df[df['SA_Score_Binary'] == 1].copy()

print(f"Total de moléculas filtradas: {len(df_filtered)} (SA Score < {threshold})")

# Salvar o dataframe filtrado em CSV
df_filtered.to_csv(output_file_filtrado, index=False)

print(f"Arquivo filtrado salvo em: {output_file_filtrado}")